# Inspect Consolidated AET Datasets

Load and visualize the two AET source datasets for January 2000.

Sources (see `catalog/variables.yml` → `aet`):
- MOD16A2 v061 (`ET_500m`, kg m-2 per 8-day composite) — local consolidated per-year NC
- SSEBop (`actual_et`, mm/month) — remote zarr at the USGS GDP STAC OSN endpoint
- MWBM (Wieczorek et al. 2024, `aet`, mm/month) — accessed from ScienceBase as a single CF-conformant NetCDF; native mm/month with `cell_methods: time: sum`. No conversion needed before comparison.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000


## Load both datasets

In [ ]:
datasets = {
    "MOD16A2 v061 (ET_500m)": {
        "path": DATASTORE / "mod16a2_v061" / f"mod16a2_v061_{TARGET_YEAR}_consolidated.nc",
        "var": "ET_500m",
        "units": "kg m-2 / 8-day",
    },
    "SSEBop (et)": {
        "zarr_url": "s3://mdmf/gdp/ssebopeta_monthly.zarr/",
        "storage_options": {
            "anon": True,
            "endpoint_url": "https://usgs.osn.mghpcc.org/",
        },
        "var": "et",
        "units": "mm/month",
    },
    "MWBM (ClimGrid, aet)": {
        "path": DATASTORE / "mwbm_climgrid" / "ClimGrid_WBM.nc",
        "var": "aet",
        "units": "mm/month",
    },
}


def _open(info):
    if "zarr_url" in info:
        return xr.open_zarr(
            info["zarr_url"],
            storage_options=info.get("storage_options", {}),
            consolidated=True,
        )
    return xr.open_dataset(info["path"])


opened = {}
for label, info in datasets.items():
    try:
        if "path" in info and not info["path"].exists():
            print(f"SKIP {label}: {info['path']} not found (run fetch first)")
            continue
        ds = _open(info)
    except Exception as exc:
        print(f"SKIP {label}: open failed: {exc}")
        continue
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")


## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)


## Plot January 2000

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, n, figsize=(8 * n, 6))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        da = ds[var].sel(time=TARGET_TIME, method="nearest")
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap="YlOrRd", robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(opened), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"AET — nearest to {TARGET_TIME}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


## Normalized comparison (mm/month, common extent)

Aggregate both datasets to **mm/month** and clip to their **common geographic extent** for a like-for-like visual comparison.

`mm/month` is already an area-normalized unit (depth per unit area per time), so pixel size does not affect the values.
The key comparability problem is that MOD16A2 is global while SSEBop is CONUS-only: without extent clipping, the shared colour scale is dominated by high-ET tropical values and the CONUS pattern is compressed for both panels.

| Source | Native units | Conversion |
|--------|-------------|------------|
| MOD16A2 `ET_500m` | kg m⁻² / 8-day composite | Weighted sum of 8-day windows overlapping the target month (1 kg m⁻² = 1 mm), then clipped to common extent |
| SSEBop `et` | mm / month | Already monthly — select nearest time step; defines the common extent |

MOD16A2 composites that straddle a month boundary are weighted by the fraction of their 8-day window falling inside the target month.


In [ ]:
import calendar
from datetime import date, timedelta

import numpy as np
import pandas as pd


def _mod16a2_monthly_mm(ds, var, year, month):
    """
    Aggregate MOD16A2 8-day composites to a monthly total in mm/month.

    Each composite timestamp is the start of an 8-day window.  Weight each
    composite by the fraction of its 8-day window that falls within the target
    month, then sum.  kg m-2 / 8-day == mm / 8-day, so weight * da gives mm
    contributed to the month from that composite.
    """
    month_start = date(year, month, 1)
    month_end = date(year, month, calendar.monthrange(year, month)[1])
    times = pd.to_datetime(ds.time.values)

    weighted_sum = None
    for t in times:
        comp_start = t.date()
        comp_end = comp_start + timedelta(days=7)  # inclusive last day of 8-day window
        overlap_start = max(comp_start, month_start)
        overlap_end = min(comp_end, month_end)
        if overlap_start > overlap_end:
            continue
        days_overlap = (overlap_end - overlap_start).days + 1
        weight = days_overlap / 8.0
        da = ds[var].sel(time=t, method="nearest")
        contribution = weight * da
        weighted_sum = contribution if weighted_sum is None else weighted_sum + contribution
    return weighted_sum  # mm/month, or None if no composites overlap the month


def _clip_to_common_extent(arrays):
    """
    Clip all arrays to the lon/lat bounding box of their intersection.
    Handles both ascending and descending latitude coordinates.
    """
    def _lon_lat_dims(da):
        lon_dim = "lon" if "lon" in da.dims else "x"
        lat_dim = "lat" if "lat" in da.dims else "y"
        return lon_dim, lat_dim

    def _bounds(da):
        lon_dim, lat_dim = _lon_lat_dims(da)
        lons = da[lon_dim].values
        lats = da[lat_dim].values
        return (lons.min(), lons.max(), lats.min(), lats.max())

    bounds = [_bounds(a) for a in arrays.values()]
    lon_min = max(b[0] for b in bounds)
    lon_max = min(b[1] for b in bounds)
    lat_min = max(b[2] for b in bounds)
    lat_max = min(b[3] for b in bounds)
    print(f"Common extent: lon [{lon_min:.2f}, {lon_max:.2f}]  lat [{lat_min:.2f}, {lat_max:.2f}]")

    clipped = {}
    for label, da in arrays.items():
        lon_dim, lat_dim = _lon_lat_dims(da)
        lat_vals = da[lat_dim].values
        # Slice direction must match coordinate order
        lat_slice = (
            slice(lat_max, lat_min) if lat_vals[0] > lat_vals[-1]  # descending
            else slice(lat_min, lat_max)
        )
        clipped[label] = da.sel({lon_dim: slice(lon_min, lon_max), lat_dim: lat_slice})
    return clipped


# Derive target year/month from the shared TARGET_TIME variable
_ts = pd.Timestamp(TARGET_TIME)
_target_year = _ts.year
_target_month = _ts.month
_month_label = f"{_target_year}-{_target_month:02d}"

# Label of the dataset that provides the land mask / colour scale reference
_SCALE_REF = "SSEBop (et)"

if opened:
    norm_arrays = {}
    for label, (ds, info) in opened.items():
        var = info["var"]
        units = info["units"]
        if "8-day" in units:
            arr = _mod16a2_monthly_mm(ds, var, _target_year, _target_month)
            if arr is None:
                print(f"SKIP {label}: no 8-day composites overlap {_month_label}")
                continue
        else:
            arr = ds[var].sel(time=TARGET_TIME, method="nearest")
        norm_arrays[label] = arr

    if len(norm_arrays) > 1:
        norm_arrays = _clip_to_common_extent(norm_arrays)

    for label, arr in norm_arrays.items():
        print(f"{label}: mean = {float(arr.mean(skipna=True)):.2f} mm/month  (common extent)")

    if norm_arrays:
        # Derive colour scale from SSEBop (land-masked CONUS) to exclude MOD16A2 ocean values
        ref_arr = norm_arrays.get(_SCALE_REF, next(iter(norm_arrays.values())))
        ref_vals = ref_arr.values.ravel()
        ref_vals = ref_vals[~np.isnan(ref_vals)]
        vmin = float(np.percentile(ref_vals, 2))
        vmax = float(np.percentile(ref_vals, 98))
        print(f"Colour scale from {_SCALE_REF}: {vmin:.2f} – {vmax:.2f} mm/month")

        fig, axes = plt.subplots(1, len(norm_arrays), figsize=(16, 6), squeeze=False)
        for idx, (label, arr) in enumerate(norm_arrays.items()):
            ax = axes[0, idx]
            arr.plot(ax=ax, cmap="YlOrRd", vmin=vmin, vmax=vmax)
            ax.set_title(f"{label}\n{_month_label} | mm/month", fontsize=11)
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")

        fig.suptitle(
            f"AET — mm/month, common extent — {_month_label}\n"
            f"colour scale from {_SCALE_REF} (land mask)",
            fontsize=13, y=1.03,
        )
        plt.tight_layout()
        plt.show()
else:
    print("No datasets available yet.")


### Why does MOD16A2 show more ET in the western US than SSEBop?

The two products use fundamentally different algorithms, and the difference is largest in arid/semi-arid regions.

**Algorithm contrast**

- **MOD16A2** uses a Penman-Monteith process model driven by atmospheric demand: VPD, radiation, LAI, and MERRA-2 meteorology. Water stress is *inferred* from VPD and stomatal conductance parameters.
- **SSEBop** uses a simplified surface energy balance driven by MODIS land surface temperature (LST) relative to cold/hot references. Water stress is *directly observed* — stressed vegetation heats up, reducing the ET fraction.

**Why the western US diverges**

1. **Demand-side overestimation.** Penman-Monteith computes potential ET from high solar radiation and large vapour pressure deficits, then scales by vegetation LAI and a stomatal conductance function. In water-limited environments (Great Basin, Colorado Plateau, southern Rockies) the atmospheric demand is very high, and if soil moisture stress is underrepresented in the MERRA-2 inputs, the model overestimates actual ET.

2. **LST captures stress directly.** SSEBop's thermal-based approach does not rely on reanalysis soil moisture; dry, water-stressed surfaces heat above the cold reference temperature and automatically receive a low ET fraction. This makes SSEBop more responsive to actual water availability in arid regions.

3. **Sparse vegetation land cover.** Sagebrush steppe and desert shrublands cover large parts of the interior West. MOD16A2 was developed and validated primarily in mesic forest/cropland systems; the Penman-Monteith parameterisation for sparse canopies is less constrained.

4. **Snow sublimation in January.** The western mountains have deep snowpack in January. MOD16A2 includes a sublimation term that SSEBop may not represent equivalently, which can inflate MOD16A2 values at high elevation in winter months.

**Calibration target implication**

For the NHM calibration target, SSEBop is generally considered more reliable over the arid West because its LST-based signal is a direct observation rather than a modelled inference. MOD16A2 provides broader spatial coverage (including areas outside the SSEBop CONUS domain) and an independent estimate for ensemble comparison.

**Note on scale factor:** The MOD16A2 domain mean of ~4491 mm/month (vs SSEBop ~8 mm/month) over CONUS in January strongly suggests a scale factor of 0.1 has not been applied to the raw integer ET_500m values. The HDF/NetCDF `scale_factor` attribute should be checked - xarray may not apply it automatically if the variable was consolidated without decoding.


## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()
